<a href="https://colab.research.google.com/github/Yurnero2706/DataAlgo-UT/blob/main/DataAlgo2026_R08.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# データ構造とアルゴリズム 第8週課題 (2026/06/10 ver.A)

---


★下記３要素を**必ず直接書き換えてから**提出すること．

* 学籍番号： 202318032
* 氏名： Nguyen Cong Nguyen
* Colabアカウント： ncnguyencva@gmail.com

**学生同士で教えた・教わった・グループワークをした場合，必ず下記の当該行を直接書き換えて記述すること．**  
申告があった場合は，論述などで内容が似ていたり，プログラムが似ていても減点はしない．（コピペレベルの同一文などは処罰対象）  
**教えた側は加点対象となる．**

**[教えた側]**  
教えた相手：＜氏名＞　＜学籍番号＞  
（何名いてもよい；教えた相手の内容が浅くなってないか確認すること）

**[教わった側]**  
教わった相手：＜氏名＞　＜学籍番号＞  
（何名いてもよい；必ず教えた側に自分の名前を書いてもらうこと）

**[グループワーク]**  
一緒に行った相手：＜氏名＞　＜学籍番号＞  
（何名いてもよいが，必ずお互いの名前を全員記すこと）

---
# 必須課題8A 騎士の巡回問題での全解探索法の計算量  

knighttour-bluteforce_J プログラムの時間計算量と空間計算量を議論せよ．盤のマスの数をNとしてよい．



Let $N$ be the total number of cells on the board (so for an 8×8 board, $N = 64$).

The brute-force program enumerates **every possible sequence of $N - 1$ knight moves** starting from the given start cell, without checking any constraint during the search. Each knight has 8 possible direction, thus the recursion complexity is $8^N$

The recursion is `knighttouronestep_bf`. Inside each call (at every node of the tree, not just leaves), the program calls `board_from_path(n_move)`, which:

- clears the board ($O(N)$ work),
- replays all moves so far to rebuild the board ($O(N)$ work in the worst case).

So total time complexity is $O(N \cdot 8^{N})$

**Space complexity — $O(N)$**

Auxiliary memory:

- `kpath[N]` — records the chosen direction for each move. $O(N)$.
- `board[N]` — the $N$-cell board. $O(N)$.
- Recursion stack — depth at most $N - 1$, each frame holds constant data (`cpt`, `d`). $O(N)$.
- Scalars (`boardsize`, `spt`, counters): $O(1)$.

So auxiliary space is $O(N)$.


---
# 必須課題8B 騎士の巡回問題でのバックトラック法での解候補の必要条件の実装

DataAlgo_UT(011)_KnightTour.ipynb で示した knighttour-backtrack_J プログラムにおいて，「チェスの8x8の盤の中で」「同じマスにを2度通ることなしに（63手かけて）」「ナイトの移動方法のみで」「塗りつぶす」ことを確認している実装部分をそれぞれ示せ．

Each of the four constraints is enforced by a specific part of `knighttour-backtrack_J.c`. The decisive section is the body of the recursive function `knighttouronestep_bt()`:

```c
// 解探索
for (d = 0; d < M; d++) {                            // (C) try each of the M=8 knight moves
    npt.x = cpt.x + move[d].x;                       // (A) knight-move offset for direction d
    npt.y = cpt.y + move[d].y;
    if (npt.x >= 0 && npt.x < boardsize.x &&         // (B) board boundary check
        npt.y >= 0 && npt.y < boardsize.y &&
        board[boardindex(npt)] == 0) {               // (D) no-revisit check
        board[boardindex(npt)] = n_move + 1;         // (E) place this move on the board
        knighttouronestep_bt(npt, n_move + 1);       // (F) recurse: depth-first
        board[boardindex(npt)] = 0;                  // (G) backtrack: undo the mark
    }
}
```

And the terminal check earlier in the same function:

```c
if (n_move == boardsize.x * boardsize.y) {           // (H) all cells visited → answer
    knighttour++;
    ...
    return;
}
```

**Mapping each constraint to the code:**

1. **"Within the 8×8 chess board"**:
   `npt.x >= 0 && npt.x < boardsize.x && npt.y >= 0 && npt.y < boardsize.y`. If the next position falls outside the board, the `if` fails and we skip to the next direction. This guarantees the knight never leaves the board.

2. **"No cell visited twice"**:
   `board[boardindex(npt)] == 0`. Each cell stores the move number when it was first visited (1 for the start cell, then 2, 3, …). A non-zero value means "already visited", so this `if` rejects revisits. The 63 moves = $N - 1$ moves part is enforced together with **(H)**.

3. **"Fill the whole board"**:
   `if (n_move == boardsize.x * boardsize.y)`. The recursion increments `n_move` each time a cell gets filled. When `n_move` reaches $\text{boardsize.x} \cdot \text{boardsize.y}$ (= 64 on an 8×8 board), the board is fully covered, so this is recorded as a solution. Combined with constraint 2 (no revisits), reaching this point automatically means all 64 cells are filled with distinct move numbers 1…64 — exactly what "63-move knight tour filling the board" means.


---
# 必須課題8C 騎士の巡回問題での実行速度の改善  

DataAlgo_UT(011)_KnightTour.ipynb で示した knighttour-bluteforce_J プログラムと knighttour-backtrack_J プログラムでは実行速度が問題になる．そこで，gccのコンパイルオプションについて調べ，実際にそれが実行速度にどのような違いをもたらすかを調査せよ．調査は実際にプログラムを実行させて行うこと．実行時間が1秒以上かかるような事例で調査すること（計測は授業内で示した方法で行うこと．実行時間が短い場合は５回以上実行して平均をとること）．

I used `knighttour-backtrack_J` as the base program because the brute-force version takes too long to finish for any board larger than $3\times3$. I ran the backtrack program on a $5\times5$ board starting from corner $(0, 0)$ with `verboselevel = 0`. I had tried to increase the size of the board to $6\times6$, but it the time complexity increased exponentially that it took longer than hours to finish 1 execution. So I had to use a $5 \times 5$ board for this execution.

For each compile option, I recompiled the program from scratch and ran:

```bash
!time ./knighttour-backtrack_J 5 5 0 0 0
```

I used the `user` time reported by the `time` command. When a run finished in under 1 second, I repeated it 5 times and took the average.

**Compile options I tested.**

- **(none)** — no optimisation at all (this is the same as `-O0`). The compiler does a literal translation of the source code.
- **`-O1`** — basic optimisations. The compiler enables transformations that are cheap to apply and almost always make the code faster.
- **`-O2`** — more optimisations, including dead code elimination, instruction scheduling, smarter register allocation. This is the default Kameda-sensei uses in the notebook.
- **`-O3`** — aggressive optimisation: function inlining, loop unrolling, auto-vectorisation.


- `gcc -O0` — baseline. user time = `0m0.181s`.
- `gcc -O1` — user time = `0m0.067s`.
- `gcc -O2` — user time = `0m0.044s`.
- `gcc -O3` — user time = `8m36.402s`.


---
# 必須課題8D N-Queens問題での斜め方向の衝突判定

DataAlgo_UT(012)_EightQueens.ipynb で示した NQueens_Jプログラムにおいて，列xの行yに女王があるとき，右下がり方向に取り合いになる（衝突し得る）女王はすべて x+y が同じになり，しかもその値が 0 から　2N-2 の間に収まることを示せ．同じく，右上がり方向に取り合いになる（衝突し得る）女王はすべて x-y+N-1 が同じで，かつその値は 0 から 2N-2 までの間に収まることを示せ．なお，ここで右下がりとは，X軸正を右に，Y軸正を上に取った場合である．

A queen at $(x, y)$ attacks four directions: horizontal, vertical, and two diagonals. The two diagonals are:

- **Right-down (右下がり)** — the line that goes from upper-left to lower-right. On this line, as $x$ increases by 1, $y$ *decreases* by 1 (because Y-positive is up, and the line slopes *downward*).
- **Right-up (右上がり)** — the line that goes from lower-left to upper-right. As $x$ increases by 1, $y$ increases by 1.

**All queens on the same right-down diagonal share the same value of $x + y$, and that value lies in $[0, 2N-2]$.**

*Proof:*

Suppose two queens sit at $(x_1, y_1)$ and $(x_2, y_2)$ on the same right-down diagonal. By the definition of right-down, moving from one queen to the other on the diagonal means
$$x_2 - x_1 \;=\; -(y_2 - y_1).$$
Rearranging gives
$$x_1 + y_1 \;=\; x_2 + y_2.$$
So $x + y$ is constant along the diagonal — every queen on that line has the same value.

Since $0 \leq x \leq N-1$ and $0 \leq y \leq N-1$, the minimum of $x + y$ is $0 + 0 = 0$ (top-left corner using the X-right/Y-up convention, this is actually the lower-left geometrically, but the algebra is the same) and the maximum is $(N-1) + (N-1) = 2N - 2$. So
$$x + y \;\in\; [0,\, 2N-2].$$

**All queens on the same right-up diagonal share the same value of $x - y + N - 1$, and that value lies in $[0, 2N-2]$.**

*Proof:*

Suppose two queens sit at $(x_1, y_1)$ and $(x_2, y_2)$ on the same right-up diagonal. By the definition of right-up,
$$x_2 - x_1 \;=\; y_2 - y_1,$$
which rearranges to
$$x_1 - y_1 \;=\; x_2 - y_2.$$
So $x - y$ is constant along the diagonal.

$x - y$ ranges over $[-(N-1),\, N-1]$ can be negative ($x < y$). To use it as an array index in C, we need to shift it by $N - 1$:
$$x - y + N - 1 \;\in\; [0,\, 2N - 2].$$
The minimum $(x - y + N - 1) = 0$ happens at $(0, N-1)$ (top-left corner with our axis convention), and the maximum $= 2N - 2$ happens at $(N-1, 0)$ (bottom-right corner).


---
# 必須課題8E 数独問題の難易度と実行時間との関係
ナンバープレースについて，出題サイト等に行って，人間向けの難易度の異なる問題を3つ以上用意し，それぞれで DataAlgo_UT(013)_NumberPlace.ipynb の NumberPlace_Jプログラムを用いて実行時間を計測して，問題の難易度と実行時間との関係を考察せよ．そのような関係になる理由についても考察すること．


(議論はここに記述）

---
---
# 発展課題8X X-SUDOKU  
2本の対角線においても，1-9の数字を1つずつ配置するという制約を増やした問題をX-SUDOKUという．例えば [こういうサイト](http://www.sudoku-space.com/x-sudoku/) などに出題がある．  
DataAlgo_UT(013)_NumberPlace.ipynb の NumberPlace_Jプログラムを改造してX-SUDOKUを解くプログラムを作成せよ．

In [7]:
%%writefile X-Sudoku-8X.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <sys/time.h>

#define N 9

typedef struct {int x; int y;} vec2i;

int board[N][N];
int numlockedcell = 0;

vec2i celltoexamine[N*N];
int numemptycell = 0;


int checkx[N][N];
int checky[N][N];
int checkb[N][N];

int checkd1[N];
int checkd2[N];

static inline int on_d1(int x, int y) { return (x == y); }
static inline int on_d2(int x, int y) { return (x + y == N - 1); }


int num_answer = 0;
int verboselevel = 0;

int xy2b(int x, int y){ return ((x / 3) + (y / 3) * 3); }

void showboard(void){
    vec2i pt;
    for (pt.y = 0; pt.y < N; pt.y++) {
        printf("%d ", pt.y);
        for (pt.x = 0; pt.x < N; pt.x++) {
            if (board[pt.x][pt.y] != 0) printf("%d", board[pt.x][pt.y]);
            else printf(".");
        }
        printf("\n");
    }
}

int readboard(char *filename){
    FILE *fd;
    vec2i pt;
    if (strcmp(filename, "-") == 0) fd = stdin;
    else if ((fd = fopen(filename, "r")) == NULL) {
        printf("readboard: cannot open %s\n", filename);
        return -1;
    }
    for (pt.y = 0; pt.y < N; pt.y++) {
        for (pt.x = 0; pt.x < N; pt.x++) {
            if (fscanf(fd, "%d", &(board[pt.x][pt.y])) != 1) {
                printf("readboard: error at (x,y) = (%d, %d)\n", pt.x, pt.y);
                return -2;
            }
            if (board[pt.x][pt.y] < 1 || board[pt.x][pt.y] > 9) board[pt.x][pt.y] = 0;
            if (board[pt.x][pt.y] == 0)
                celltoexamine[numemptycell++] = pt;
        }
    }
    fclose(fd);
    showboard();
    printf("readboard: empty cells = %d\n", numemptycell);
    return 0;
}

void sudokuonestep(int pos){
    int x, y, b;
    int f;

    if (pos == numemptycell) {
        if (verboselevel >= 1) printf("======== %d ========\n", num_answer);
        if (verboselevel >= 2) showboard();
        num_answer++;
        return;
    }

    x = celltoexamine[pos].x;
    y = celltoexamine[pos].y;
    b = xy2b(x, y);

    for (f = 1; f <= N; f++) {
        // standard three checks (column, row, 3x3 block)
        if (checkx[x][f-1] || checky[y][f-1] || checkb[b][f-1]) continue;

        if (on_d1(x, y) && checkd1[f-1]) continue;
        if (on_d2(x, y) && checkd2[f-1]) continue;

        checkx[x][f-1] = 1;
        checky[y][f-1] = 1;
        checkb[b][f-1] = 1;
        if (on_d1(x, y)) checkd1[f-1] = 1;
        if (on_d2(x, y)) checkd2[f-1] = 1;
        board[x][y] = f;

        sudokuonestep(pos + 1);

        checkx[x][f-1] = 0;
        checky[y][f-1] = 0;
        checkb[b][f-1] = 0;
        if (on_d1(x, y)) checkd1[f-1] = 0;
        if (on_d2(x, y)) checkd2[f-1] = 0;
        board[x][y] = 0;
    }
}

int main(int argc, char *argv[]){
    int r = 0;
    int i, f;
    int x, y, b;
    struct timeval ts, te;

    if (argc != 3) {
        printf("Usage: %s <problem_file> <verbose_level>\n", argv[0]);
        return -1;
    }

    if ((r = readboard(argv[1])) != 0) {
        printf("Error: file reading status : %d\n", r);
        return r;
    }
    verboselevel = atoi(argv[2]);

    for (i = 0; i < N; i++) {
        for (f = 0; f < N; f++)
            checkx[i][f] = checky[i][f] = checkb[i][f] = 0;
        checkd1[i] = 0;
        checkd2[i] = 0;
    }

    for (x = 0; x < N; x++) {
        for (y = 0; y < N; y++) {
            if (board[x][y] != 0) {
                f = board[x][y] - 1;
                b = xy2b(x, y);
                checkx[x][f] = 1;
                checky[y][f] = 1;
                checkb[b][f] = 1;
                if (on_d1(x, y)) checkd1[f] = 1;
                if (on_d2(x, y)) checkd2[f] = 1;
            }
        }
    }

    gettimeofday(&ts, NULL);
    sudokuonestep(0);
    gettimeofday(&te, NULL);

    printf("===================\n");
    printf("Number of the answers: %d\n", num_answer);
    printf("Time = %.6f [sec]\n",
        (float)(te.tv_sec - ts.tv_sec) + (te.tv_usec - ts.tv_usec) / 1000000.0);
    return 0;
}


Overwriting X-Sudoku-8X.c


In [10]:
%%writefile X-sudoku-A.txt
0 0 0  0 0 0  7 0 0
0 0 4  0 0 0  0 6 0
0 2 0  0 0 9  0 9 0

0 0 0  0 0 0  0 8 5
5 0 1  9 2 0  0 0 0
3 0 8  5 0 7  0 1 0

0 0 0  0 0 0  5 0 7
0 0 0  0 0 0  0 0 0
9 0 0  6 0 0  0 3 1

Overwriting X-sudoku-A.txt


In [11]:
!echo "コンパイルと実行コマンドをここに（提出時に出力をつけておくこと，例はX-sudoku-A.txtでよい）"
!gcc -Wall -O2 -o X-Sudoku-8X X-Sudoku-8X.c
!./X-Sudoku-8X X-sudoku-A.txt 2

コンパイルと実行コマンドをここに（提出時に出力をつけておくこと，例はX-sudoku-A.txtでよい）
0 ......7..
1 ..4....6.
2 .2...9.9.
3 .......85
4 5.192....
5 3.85.7.1.
6 ......5.7
7 .........
8 9..6...31
readboard: empty cells = 58
Number of the answers: 0
Time = 0.000354 [sec]


---
# 発展課題8Y 騎士の巡回問題の盤サイズへの挑戦  
騎士の巡回問題において，盤のサイズを3x3から徐々に大きくしていった時，解の数と実行時間との関係に鋤いて調査せよ．このとき，長方形についても調査してみよ．(3x3, 3x4, 3x5, 3x6, など)   
なお，8x8の実行は，本授業で提示しているアルゴリズムでは無謀である．どうして無謀と思えるのか，考察を試みよ．

---
# 課題提出法

筑波大学工学システム学類３年生向け．  
FG24711 / FG34711．  

必須課題は全て実施すること．
発展課題はしなくともよいが，A+取得には発展課題を（全課題提出を通して）1つ以上実施していることが必要条件である．

該当するmanabaに課題提出のエントリを設けるので，そこに本**Colab notebookを提出**すること．

# 出典

筑波大学工学システム学類  
データ構造とアルゴリズム  
担当：亀田能成  

---
2026/06/10 ver.A  